# Plot Batch Experiment for Several Values of alpha

In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as clr

import mars
from mars.measurements.properties import PropertyEnum

plt.rcParams.update({"font.size": 12})

In [ ]:
# Alias old package name
sys.modules["msean"] = mars

In [ ]:
# Input file
results_path = "../inputs/batch_experiment_results.pkl"
with open(results_path,"rb",) as f:
    results_all = pickle.load(f)

In [ ]:
# Output directories
plot_dir = Path("../outputs/plots")
plot_dir.parent.mkdir(parents=True, exist_ok=True)

tex_dir = Path("../outputs/tex")
tex_dir.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Consider only models obtained with best sigma value (based on fitting)
std_val = 0.175

results = {
    k[1]: v
    for k, v in results_all.items()
    if k[0] == std_val
}

In [ ]:
# Alias old package imports to MARS
for alpha, alpha_results in results.items():
    converted = {}

    for old_prop, value in alpha_results.items():
        new_prop = PropertyEnum(old_prop.value)
        converted[new_prop] = value

    results[alpha] = converted

### Plotting Style

In [ ]:
alpha_list = [(1/2)**(i/2) for i in range(0, 21)]
layer_sizes = [656, 4700, 326, 7962, 5817]

In [ ]:
linestyles = ["-"]
cmap = cm.viridis

norm = clr.SymLogNorm(
    linthresh=1e-3,
    vmin=min(alpha_list),
    vmax=max(alpha_list),
    base=2,
)

tick_is = list(range(0, 21))
ticks = [(1/2)**(i/2) for i in tick_is]
log_ticks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]

In [ ]:
prop_labels = {
    PropertyEnum.DENSITY: r"Density ($\rho$)",
    PropertyEnum.DEGREE_DISTRIBUTION: rf"Degree ($\mathit{{k}}$)",
    PropertyEnum.AVERAGE_LOCAL_CLUSTERING: rf"Average clustering coefficient ($\bar{{c}}$)",
    PropertyEnum.AVERAGE_MULTIPLEXITY: r"Mean share of multiplex ties ($\mathit{{\bar{\mu}}}$)"
}

alpha_label = r"Spatial freedom ($\alpha$)"

### Global Scalar Properties

In [ ]:
global_props = [
    PropertyEnum.DENSITY,
    PropertyEnum.AVERAGE_LOCAL_CLUSTERING,
    PropertyEnum.AVERAGE_MULTIPLEXITY,
]

In [ ]:
for prop in global_props:
    means = np.array(
        [results[a][prop][0] for a in alpha_list],
        dtype=float,
    )

    fig, ax = plt.subplots()
    ax.plot(
        alpha_list,
        means,
        linewidth=2,
        linestyle="--",
        marker="o",
    )

    ax.set_xscale("log", base=2)
    ax.set_xlabel(r"Spatial freedom ($\alpha$)")
    label = prop_labels.get(prop, prop.name.lower())
    ax.set_ylabel(label)

    plt.savefig(
        plot_dir / f"{prop.name.lower()}_vs_alpha-log.png",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

In [ ]:
# Will be used in closure plots
share_multiplex_ties_list = np.array([results[a][PropertyEnum.AVERAGE_MULTIPLEXITY][0] for a in alpha_list])

### Mean Average Alter Distance

In [ ]:
prop = PropertyEnum.AVERAGE_ALTER_DISTANCE_DISTRIBUTION
fig, ax = plt.subplots()

mean_alter_distances = []
for alpha in alpha_list:
    dist = results[alpha][prop]

    x = dist[:, 0].astype(float)
    y = dist[:, 1].astype(float)

    # Mean of the distribution
    mean_alter_distance = np.sum(x * y) / np.sum(y)
    mean_alter_distances.append(mean_alter_distance)

ax.plot(
    alpha_list,
    mean_alter_distances,
    linewidth=2,
    linestyle="--",
    marker="o",
)

ax.set_xscale("log", base=2)
ax.set_xlabel(r"Spatial freedom ($\alpha$)")
ax.set_ylabel(r"Mean average alter distance ($\bar{d}_{alter}$)")

plt.savefig(
    plot_dir / "mean_average_alter_distance_vs_alpha.png",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Will be used in closure plots TODO: change var name later on so it's not doubled
avg_prop_alter_dist = mean_alter_distances

### Degree Distribution

In [ ]:
def rebin_degree_distribution(x, y, se, bin_width=5):
    x = np.asarray(x)
    y = np.asarray(y)
    se = np.asarray(se)

    bin_id = (x // bin_width).astype(int)

    x_binned = []
    y_binned = []
    se_binned = []

    for b in np.unique(bin_id):
        mask = bin_id == b

        x_binned.append(np.mean(x[mask]))
        y_binned.append(np.nansum(y[mask]))
        se_binned.append(np.sqrt(np.nansum(se[mask] ** 2)))

    return np.array(x_binned), np.array(y_binned), np.array(se_binned)

In [ ]:
# Skip half-integer exponents
integer_mask = np.array(tick_is) % 2 == 0
integer_ticks = np.array(ticks)[integer_mask]
integer_tick_is = np.array(tick_is)[integer_mask]
highlight_idx = 18  # alpha = 2^-9

fig, ax = plt.subplots(figsize=(5, 5))

for i, (alpha, exponent_i) in enumerate(zip(alpha_list, tick_is)):
    if exponent_i % 2 != 0:
        continue

    dist = results[alpha][PropertyEnum.DEGREE_DISTRIBUTION]
    x = dist[:, 0]
    y = dist[:, 1].astype(float)
    se = dist[:, 3].astype(float)

    x, y, _ = rebin_degree_distribution(
        x,
        y,
        se,
        bin_width=10,
    )
    y[y <= 0] = np.nan

    color = cmap(norm(alpha))

    ax.plot(
        x,
        y,
        color=color,
        linestyle=linestyles[i % len(linestyles)],
        linewidth=1.2,
        alpha=1.0,
        zorder=3,
    )

sm = cm.ScalarMappable(
    cmap=cmap,
    norm=norm,
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(alpha_label)
cbar.set_ticks(integer_ticks)
cbar.set_ticklabels([
    rf"$2^{{{-int(exponent/2)}}}$"
    for exponent in integer_tick_is
])

label = prop_labels.get(PropertyEnum.DEGREE_DISTRIBUTION, PropertyEnum.DEGREE_DISTRIBUTION.name.lower())

ax.set_xlabel(label)
ax.set_ylabel("Frequency")
ax.set_xscale("log")
ax.set_yscale("log")

plt.savefig(
    plot_dir / f"{PropertyEnum.DEGREE_DISTRIBUTION.name.lower()}_all_alpha_log_binned_10_emph.png",
    bbox_inches="tight",
)

plt.show()

### Closure plots

In [ ]:
plt.rcParams.update({"font.size": 10})

In [ ]:
# Prepare the shared data
avg_alter_dist = np.asarray(avg_prop_alter_dist, dtype=float)
multiplexity = np.asarray(share_multiplex_ties_list, dtype=float)

# Average local clustering
clustering_prop = PropertyEnum.AVERAGE_LOCAL_CLUSTERING
clustering_means = np.array([
    results[alpha][clustering_prop][0]
    for alpha in alpha_list
], dtype=float)

# 3D triangle ratio
num_3d_means = []

for alpha in alpha_list:
    mean_dims, _, _ = results[alpha][PropertyEnum.TRIANGLE_DIMENSIONS]
    _, _, tau_3d = mean_dims
    num_3d_means.append(tau_3d)

num_3d_means = np.asarray(num_3d_means, dtype=float)

In [ ]:
# Create 2 x 2 tiled figure
fig, axes = plt.subplots(
    2,
    2,
    figsize=(9, 5),
    sharex="col",
    sharey="row",
    constrained_layout=True,
)

ax_tl = axes[0, 0]
ax_tr = axes[0, 1]
ax_bl = axes[1, 0]
ax_br = axes[1, 1]

# Top left: clustering vs average alter distance

ax_tl.plot(
    avg_alter_dist,
    clustering_means,
    linewidth=1.4,
    linestyle="--",
    marker="o",
)
ax_tl.set_ylabel(
    prop_labels.get(
        clustering_prop,
        clustering_prop.name.lower(),
    )
)

# Top right: clustering vs multiplexity
ax_tr.plot(
    multiplexity,
    clustering_means,
    linewidth=1.4,
    linestyle="--",
    marker="o",
)

# Bottom left: 3D triangle ratio vs average alter distance
ax_bl.plot(
    avg_alter_dist,
    num_3d_means,
    linewidth=1.4,
    linestyle="--",
    marker="o",
)

ax_bl.set_xlabel(r"Mean average alter distance (${\bar{{d}}_{alter}}$)")
ax_bl.set_ylabel(r"Number of 3D triangles ($\tau_{3D}$)")

# Bottom right: 3D triangle ratio vs multiplexity
ax_br.plot(
    multiplexity,
    num_3d_means,
    linewidth=1.4,
    linestyle="--",
    marker="o",
)

ax_br.set_xlabel(r"Mean share of multiplex ties ($\bar{{\mu}}$)")
fig.align_ylabels([ax_tl, ax_bl])
ax_br.invert_xaxis()

fig.savefig(
    plot_dir / "clustering_triangle_ratio_vs_mobility_overlap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

### Summary table

In [ ]:
# Build table
rows = []
for alpha in sorted(results.keys(), key=float):

    # Density
    density = results[alpha][PropertyEnum.DENSITY][0]

    # Mean share of multiplex ties
    mean_multiplexity = results[alpha][
        PropertyEnum.AVERAGE_MULTIPLEXITY
    ][0]

    # Average local clustering coefficient
    average_local_clustering = results[alpha][
        PropertyEnum.AVERAGE_LOCAL_CLUSTERING
    ][0]

    # Mean average alter distance
    alter_distance_distribution = np.asarray(
        results[alpha][
            PropertyEnum.AVERAGE_ALTER_DISTANCE_DISTRIBUTION
        ],
        dtype=float,
    )

    distance_values = alter_distance_distribution[:, 0]
    distance_weights = alter_distance_distribution[:, 1]

    mean_alter_distance = np.average(
        distance_values,
        weights=distance_weights,
    )

    # Number of 1D and 3D triangles
    mean_triangle_dimensions = results[alpha][
        PropertyEnum.TRIANGLE_DIMENSIONS
    ][0]

    number_1d_triangles = mean_triangle_dimensions[0]
    number_3d_triangles = mean_triangle_dimensions[2]

    # Alpha label
    exponent = np.log2(alpha)

    alpha_label = rf"$2^{{{exponent:g}}}$"

    rows.append({
        "alpha": alpha_label,
        "density": density,
        "mean_average_alter_distance": mean_alter_distance,
        "mean_share_multiplex_ties": mean_multiplexity,
        "average_local_clustering_coefficient":
            average_local_clustering,
        "number_1d_triangles": number_1d_triangles,
        "number_3d_triangles": number_3d_triangles,
    })


table = pd.DataFrame(rows)

In [ ]:
# Format triangle counts in scientific notation
for column in ["number_1d_triangles", "number_3d_triangles"]:
    table[column] = table[column].apply(
        lambda value: (
            rf"${value / 10**int(np.floor(np.log10(abs(value)))):.3f}"
            rf" \times 10^{{{int(np.floor(np.log10(abs(value))))}}}$"
            if value != 0
            else r"$0.000$"
        )
    )


In [ ]:
# LaTeX column names
table = table.rename(
    columns={
        "alpha":
            r"\makecell{Spatial freedom\\($\alpha$)}",
        "density":
            r"\makecell{Density\\($\rho$)}",
        "mean_average_alter_distance":
            r"\makecell{Mean average\\alter distance\\"
            r"($\bar{d}_{\mathrm{alter}}$)}",
        "mean_share_multiplex_ties":
            r"\makecell{Mean share of\\multiplex ties\\"
            r"($\bar{\mu}$)}",
        "average_local_clustering_coefficient":
            r"\makecell{Average clustering\\coefficient\\"
            r"($\bar{c}$)}",
        "number_1d_triangles":
            r"\makecell{Number of\\1D triangles\\"
            r"($\tau_{1D}$)}",
        "number_3d_triangles":
            r"\makecell{Number of\\3D triangles\\"
            r"($\tau_{3D}$)}",
    }
)

In [ ]:
# Generate and save LaTeX
latex_table = table.to_latex(
    index=False,
    escape=False,
    column_format="lrrrrrr",
    float_format=lambda value: f"{value:.4f}",
    label="tab:network_properties_by_alpha",
)

output_path = tex_dir / "network_properties_by_alpha.tex"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(latex_table,encoding="utf-8")

print(latex_table)